<a href="https://colab.research.google.com/github/Ankitha-HR/rag-study-assistant/blob/main/rag_study_assistant.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# RAG Study Assistant

Ask questions about PDF documents and get answers **with page citations**.
If the answer is not in the documents, the assistant says so instead of guessing.

**Tech stack:** Python, LangChain, ChromaDB, Sentence-Transformers (all-MiniLM-L6-v2), Google Gemini API

**How it works**

```
PDFs -> Chunking -> Embeddings -> Vector DB (ChromaDB)
Question -> Embedding -> Top-k similar chunks -> Prompt -> Gemini -> Answer + sources
```



## Step 1: Load the PDFs and split them into chunks

In [7]:
!pip install -q langchain langchain-community langchain-text-splitters chromadb sentence-transformers pypdf

In [8]:
import os, urllib.request
from langchain_community.document_loaders import PyPDFDirectoryLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

# Download two sample papers (RAG and "Attention Is All You Need")
os.makedirs("data", exist_ok=True)
papers = {
    "rag_paper.pdf": "https://arxiv.org/pdf/2005.11401",
    "attention_paper.pdf": "https://arxiv.org/pdf/1706.03762",
}
for name, url in papers.items():
    path = os.path.join("data", name)
    if not os.path.exists(path):
        req = urllib.request.Request(url, headers={"User-Agent": "Mozilla/5.0"})
        with urllib.request.urlopen(req) as r, open(path, "wb") as f:
            f.write(r.read())

# Load every page, then cut the pages into small overlapping chunks
docs = PyPDFDirectoryLoader("data").load()
splitter = RecursiveCharacterTextSplitter(chunk_size=800, chunk_overlap=150)
chunks = splitter.split_documents(docs)

print("Pages:", len(docs), "-> Chunks:", len(chunks))

Pages: 34 -> Chunks: 176


## Step 2: Embed the chunks and store them in a vector database

In [9]:
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import Chroma

embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")

# Start from a clean collection so re-running this cell never creates duplicates
try:
    Chroma(collection_name="rag_demo", embedding_function=embeddings).delete_collection()
except Exception:
    pass

vectordb = Chroma.from_documents(documents=chunks, embedding=embeddings, collection_name="rag_demo")
print("Stored chunks:", vectordb._collection.count())

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Stored chunks: 176


## Step 3: Test retrieval (the "R" in RAG)
Check that the right chunks come back before adding the LLM.

In [10]:
test_questions = [
    "What is multi-head attention?",
    "How does the RAG model combine a retriever with a generator?",
]

for q in test_questions:
    print("QUESTION:", q)
    for i, d in enumerate(vectordb.similarity_search(q, k=3), 1):
        print(f"  {i}. {d.metadata['source']} | page {d.metadata['page'] + 1}")
        print("     ", d.page_content[:150].replace("\n", " "))
    print()

QUESTION: What is multi-head attention?
  1. data/attention_paper.pdf | page 14
      Full attentions for head 5. Bottom: Isolated attentions from just the word ‘its’ for attention heads 5 and 6. Note that the attentions are very sharp 
  2. data/attention_paper.pdf | page 5
      dk = dv = dmodel/h = 64. Due to the reduced dimension of each head, the total computational cost is similar to that of single-head attention with full
  3. data/attention_paper.pdf | page 5
      output values. These are concatenated and once again projected, resulting in the final values, as depicted in Figure 2. Multi-head attention allows th

QUESTION: How does the RAG model combine a retriever with a generator?
  1. data/rag_paper.pdf | page 19
      input. In these cases, once retrieval had collapsed, the generator would learn to ignore the documents, and the RAG model would perform equivalently t
  2. data/rag_paper.pdf | page 2
      2 Methods We explore RAG models, which use the input sequence x to re

## Step 4: Connect to Gemini (with retry and model fallback)
Free API servers are sometimes busy (errors 503 / 429). This function waits and retries,
then switches to a backup model if needed.

In [15]:
import time
from google.colab import userdata
from google import genai
from google.genai import types

client = genai.Client(
    api_key=userdata.get("GEMINI_API_KEY"),
    http_options=types.HttpOptions(timeout=30000),   # give up on a call after 30 seconds
)

MODELS = ["gemini-3.5-flash-lite", "gemini-flash-lite-latest", "gemini-3.8-flash"]

def generate_with_retry(prompt, retries=2):
    for model in MODELS:
        for attempt in range(retries):
            try:
                text = client.models.generate_content(model=model, contents=prompt).text
                return text or "(empty response)"
            except Exception as e:
                msg = str(e)
                busy = any(x in msg for x in ["503", "429", "UNAVAILABLE", "timed out", "Timeout"])
                if busy:
                    wait = 2 ** attempt * 2
                    print(f"{model} busy, retrying in {wait}s...")
                    time.sleep(wait)
                else:
                    print(f"{model} failed: {msg[:120]}")
                    break
    return "All models are busy right now. Please try again in a minute."

## Step 5: The full RAG function
**Retrieve** the best chunks, **Augment** the prompt with them, **Generate** the answer.

In [12]:
def retrieve(question, k=6):
    return vectordb.similarity_search(question, k=k)

def build_prompt(question, docs):
    context = "\n\n".join(
        f"[Source: {d.metadata['source']}, page {d.metadata['page'] + 1}]\n{d.page_content}"
        for d in docs
    )
    return f"""You are a study assistant. Answer the question using ONLY the context below.
If the answer is not in the context, say "I couldn't find that in the documents."
Mention the source file and page number for the facts you use.

Context:
{context}

Question: {question}

Answer:"""

def ask(question, k=6):
    docs = retrieve(question, k)                               # 1. RETRIEVE
    answer = generate_with_retry(build_prompt(question, docs)) # 2. AUGMENT + 3. GENERATE
    sources = sorted({f"`{d.metadata['source']}` p.{d.metadata['page'] + 1}" for d in docs})
    return answer + "\n\n**Sources retrieved:** " + ", ".join(sources)

## Step 6: Try it

In [13]:
from IPython.display import Markdown, display

def show(question, k=6):
    display(Markdown(ask(question, k)))

show("What is the difference between RAG-Sequence and RAG-Token?")
show("What is multi-head attention?")
show("What is the capital of France?")   # not in the documents -> should refuse

gemini-3.8-flash busy, retrying in 2s...


Based on the provided context, the main differences between RAG-Sequence and RAG-Token are:

* **Document Usage during Generation:** 
  * **RAG-Sequence** uses the same retrieved document to generate the complete sequence. It treats the retrieved document as a single latent variable that is marginalized to obtain the sequence probability via a top-$K$ approximation, where the generator produces the output sequence probability for each document before they are marginalized [Source: data/rag_paper.pdf, page 3].
  * **RAG-Token** can draw a different latent document for each target token and marginalize accordingly. The generator produces a distribution for the next output token for each document, marginalizes them, and repeats the process for each subsequent token. This allows the generator to select and combine content from several documents when producing an answer [Source: data/rag_paper.pdf, page 3, page 6].

* **Decoding / Inference:**
  * At test time, RAG-Sequence and RAG-Token require different methods to approximate $\arg\max_y p(y|x)$ [Source: data/rag_paper.pdf, page 4].

*(Note: In sequence classification tasks where the target sequence has a length of one, RAG-Sequence and RAG-Token are equivalent) [Source: data/rag_paper.pdf, page 3].*

**Sources retrieved:** `data/rag_paper.pdf` p.18, `data/rag_paper.pdf` p.3, `data/rag_paper.pdf` p.4, `data/rag_paper.pdf` p.6, `data/rag_paper.pdf` p.8

Based on the provided context, **multi-head attention** is an attention mechanism that allows the model to jointly attend to information from different representation subspaces at different positions, preventing the loss of information that occurs when a single attention head averages these positions [Source: data/attention_paper.pdf, page 5]. It is used to counteract the reduced effective resolution caused by averaging attention-weighted positions [Source: data/attention_paper.pdf, page 2].

Key details of multi-head attention include:
* **Function and Projections:** It computes multiple attention heads in parallel, concatenates their outputs, and projects them:
  $$\text{MultiHead}(Q, K, V) = \text{Concat}(\text{head}_1, ..., \text{head}_h)W^O$$
  where $\text{head}_i = \text{Attention}(QW_i^Q, KW_i^K, VW_i^V)$ using parameter matrices $W_i^Q \in \mathbb{R}^{d_{\text{model}}\times d_k}$, $W_i^K \in \mathbb{R}^{d_{\text{model}}\times d_k}$, $W_i^V \in \mathbb{R}^{d_{\text{model}}\times d_v}$, and $W^O \in \mathbb{R}^{hd_v \times d_{\text{model}}}$ [Source: data/attention_paper.pdf, page 5].
* **Efficiency:** Because the dimension of each head is reduced (using $h = 8$ parallel heads and $d_k = d_v = d_{\text{model}}/h = 64$), its total computational cost is similar to that of single-head attention with full dimensionality [Source: data/attention_paper.pdf, page 5].

**Sources retrieved:** `data/attention_paper.pdf` p.13, `data/attention_paper.pdf` p.14, `data/attention_paper.pdf` p.15, `data/attention_paper.pdf` p.2, `data/attention_paper.pdf` p.5

gemini-3.8-flash busy, retrying in 2s...
gemini-3.8-flash busy, retrying in 4s...
gemini-3.8-flash busy, retrying in 8s...


I couldn't find that in the documents.

**Sources retrieved:** `data/attention_paper.pdf` p.1, `data/attention_paper.pdf` p.10, `data/attention_paper.pdf` p.14, `data/attention_paper.pdf` p.15, `data/attention_paper.pdf` p.3, `data/rag_paper.pdf` p.7

## Step 7: Evaluation
7 test questions: 5 answerable from the documents and 2 out-of-scope.
A good system answers the first 5 and refuses the last 2 (no hallucination).

In [16]:
tests = [
    ("What is multi-head attention?", "attention_paper.pdf", True),
    ("Why does the Transformer use positional encoding?", "attention_paper.pdf", True),
    ("What is retrieval-augmented generation?", "rag_paper.pdf", True),
    ("What is the difference between RAG-Sequence and RAG-Token?", "rag_paper.pdf", True),
    ("What retriever does the RAG model use?", "rag_paper.pdf", True),
    ("What is the capital of France?", None, False),
    ("Who won the 2022 FIFA World Cup?", None, False),
]

results = []
for q, expected_file, should_answer in tests:
    docs = retrieve(q, k=6)
    files = {d.metadata["source"].split("/")[-1] for d in docs}
    answer = generate_with_retry(build_prompt(q, docs))
    refused = "couldn't find" in answer.lower()
    retrieval_ok = (expected_file in files) if expected_file else None
    correct = (not refused) if should_answer else refused
    results.append(correct)
    print(f"Q: {q}\n   retrieval_ok={retrieval_ok} | correct_behaviour={correct}\n")

print(f"Correct behaviour: {sum(results)}/{len(results)}")

Q: What is multi-head attention?
   retrieval_ok=True | correct_behaviour=True

Q: Why does the Transformer use positional encoding?
   retrieval_ok=True | correct_behaviour=True

Q: What is retrieval-augmented generation?
   retrieval_ok=True | correct_behaviour=True

Q: What is the difference between RAG-Sequence and RAG-Token?
   retrieval_ok=True | correct_behaviour=True

Q: What retriever does the RAG model use?
   retrieval_ok=True | correct_behaviour=True

Q: What is the capital of France?
   retrieval_ok=None | correct_behaviour=True

Q: Who won the 2022 FIFA World Cup?
   retrieval_ok=None | correct_behaviour=True

Correct behaviour: 7/7
